# Filtering Parameter Validation — why these values, not others

The nine thresholds and hyper-parameters in the comment-filtering pipeline, swept on the real data. Each section aims to show the value is **forced by the data, not the product of performance tuning**.

**Data** — Section P (pipeline overview) uses the **full 2014-2025 final sample** (2.15M raw comments, same figures as the paper's §3.3 table). The A–G parameter sweeps use the **full 2014-2025 sample** (1.41M comments entering the toxicity filter; F/G from the real stock_score & is_stock columns, D/E from a 40k-per-year stratified sample). Strategy metrics (γ, sentiment MA — H, I) are re-derived over the full 10-year K-FGI pipeline.

**Limitation** — sentiment probabilities exist only for kept comments, so daily-sentiment stability is simulated only for the 'lower τ (stricter)' direction; for the 'raise τ (looser)' direction see B-3 (weight mass).

## 0. Setup and data load
The first `compute()` takes ~3–4 min and is cached to `_compute_cache.pkl`; later runs are instant.

In [ ]:
%matplotlib inline
import filter_param_lib as L
import matplotlib.pyplot as plt
LANG = "en"
L.use_style(LANG)

In [ ]:
tox, kept = L.load_all()

In [ ]:
R = L.compute(tox, kept)   # 최초 3~4분, 이후 캐시

## P. Filtering pipeline overview  (final 2014-2025 sample)

Comments used for sentiment analysis pass: **raw crawl → empty/duplicate removal → stage 1 political → stage 2 toxicity → stage 3 economic**. The table and charts below use the same figures as the paper's §3.3 filtering table (full 2014-2025), re-derived from the per-year detail table in `10_Paper_outputs/04_투고_전략/FINAL_REVISION_MASTER_KR.md`.

- Stage 1 political filtering removes politically framed comments by regex-matching political keywords (politician / party / event names), to stop the sentiment model from reading expressions like '재앙' (catastrophe) or '내란' (insurrection) as market-negative sentiment.
- The largest drop is stage 3 economic filtering (84.9% of the toxicity-passed set), removing ordinary comments unrelated to stocks/economy.
- **Why raw counts differ by year (P-6)**: (i) collected articles grew from 3,651 (2014) to ~7,200 from 2018 (top-10 articles per section per day), and (ii) mean non-empty comments collected per article rose from 8 (2016) to 31 (2021) — not a collection cap but genuine growth in comments per financial article (median 2→23) with the retail-investing boom. The crawler collects all same-day comments per article (Naver API limit ~200). 2018 (+124%) and 2021 (peak 311k) are the main growth years.
- **Why removal rates differ by year (P-7)**: (i) the empty (deleted) comment rate jumped to 25–37% in 2017–2019 after Naver's 2018 comment overhaul; (ii) 2019 has only 2 economic core keywords, giving the lowest final retention (2.4%) — the DLF scandal, Cho Kuk affair and Japan export controls pushed comments toward non-stock topics.

> Note — the A–G parameter sweeps run on the full 2014-2025 sample (1.41M comments); F/G use the real per-comment directly loadable. Same filter scripts and toxicity model; the conclusions (plateau ranges for τ and score, γ having no effect) are insensitive to the year range. Strategy metrics (H, I) are re-derived over the full 10-year K-FGI pipeline.

### P1

Starting from 2,150,215 raw crawled comments. Empty/duplicate removal leaves 1,575,268 (73.3%) as political-filter input; stage 1 political-keyword filtering removes 165,288 (10.5% of input), stage 2 toxicity filtering (τ=0.95) removes 131,337 (9.3% of political-passed), and stage 3 economic-relevance filtering removes 1,085,526 (84.9% of toxicity-passed). The final economy/stock comment set is 193,117 (9.0% of raw), of which 192,888 were sentiment-scored. (Same figures as the paper's §3.3 filtering table; per-year detail is in the Appendix / `10_Paper_outputs/04_투고_전략/FINAL_REVISION_MASTER_KR.md`.)

In [ ]:
L.make_fig("P1", R, LANG)

### P2

Each bar is the number of comments left after that stage: 2.15M raw → 1.58M political input → 1.41M political-passed → 1.28M toxicity-passed → 193k final. The largest drop is stage 3 economic-relevance filtering (84.9% of the toxicity-passed set), which removes the large volume of ordinary comments unrelated to stocks/economy.

In [ ]:
L.make_fig("P2", R, LANG)

### P3

Left: share of political comments by year — around 4–6% in 2014–2016, above 10% from 2018, and 15.0% in 2025 around the impeachment / martial-law period. Right: the top 16 matched political keywords (2014-2025 total). Early years are dominated by 박근혜/근혜, later years by 민주당/문재인/재명/이재명/윤석열/석열/탄핵. These politician/party/event terms inject a political framing bias into the sentiment aggregation, so they are removed.

In [ ]:
L.make_fig("P3", R, LANG)

### P4

Paper appendix table. Raw comment volume grew to 200k–310k per year from 2018; the final economy/stock comment count is highest (~30k) in years with market controversies (2021 short-selling, the 2023+ '국장' debate). The final retention rate ranges 4–13% per year, averaging 9.0%.

In [ ]:
L.make_fig("P4", R, LANG)

### P5

Each year's bar decomposed into removed/retained by stage: from the bottom, final retained (economy/stock), economic-irrelevant removed, toxicity removed, political removed, empty/duplicate. In every year the 'economic-irrelevant removed' segment (grey) is the largest.

In [ ]:
L.make_fig("P5", R, LANG)

### P6

Each day the crawler picks the top 10 articles by comment count in each of the securities and finance sections, and collects all of that day's comments for each (Naver's comment API returns at most ~200 per article; fewer than 1% of articles hit this). (Left) collected articles grew from 3,651 (2014) to ~7,200 from 2018 — in 2014–2016 there were often fewer than 10 commented financial articles to fill the top-10 per section; from 2018 the top-10 is filled every day. (Right) the mean non-empty comments collected per article rose from 8 (2016) to 31 (2021) — not because of a collection cap but because the actual comment volume per financial-news article grew (Naver's displayed comment count, median: 2 in 2016 → 23 in 2021), driven by the 2020 retail-investing boom and the 2021 short-selling controversy. Together, raw comments grew ~9× from 34k (2014) to 311k (2021).

In [ ]:
L.make_fig("P6", R, LANG)

### P7

(Left) empty-comment rate: near 0% through 2016, spiking to 25–37% in 2017–2019, then easing to ~22%. After the 2018 comment-manipulation scandal Naver overhauled its comment system and expanded its AI filter; deleted comments now remain as empty slots (valid comment_id, empty body), whereas pre-2016 deletions were fully purged and never collected. (Right) number of core keywords the economic-relevance filter extracts per year: 2019 has the fewest (2), giving the lowest final retention (2.4%). In 2019, financial-news comments concentrated on non-stock economic/social issues (the DLF mis-selling scandal, the Cho Kuk private-equity affair, the Japan export controls), so stock-discussion words fell below the 1% document-frequency threshold. Retention recovers above 10% in 2020–2021 (the retail boom) as more core keywords appear.

In [ ]:
L.make_fig("P7", R, LANG)

### P8

(Left) year-by-year document frequency (DF) of the main stock keywords the economic filter extracts. The dashed line (0.5%) is the core-keyword threshold. In 2019 only '공매도' (short selling) and '주식' (stock) clear it; '금리', '개미들', '주가', '기관' all fall below 0.5% and are demoted from core to auxiliary. Since passage requires 'score ≥ 10 and at least one core keyword', with only two core keywords a comment must literally contain '공매도' or '주식' → final retention 5.1% (vs 12–18% in other years). (Right) topic distribution of the securities/finance news crawled in 2019 (comment-weighted). Bank/DLF mis-selling, the Cho Kuk prosecution, the Japan export controls, pensions and labour dominate; actual stock-market articles are a minority — retail-investing discourse began only after March 2020, so 2019 financial-news comments were not about the stock market and the filter removed them accordingly.

In [ ]:
L.make_fig("P8", R, LANG)

## A. Toxicity weighting exponent γ  (current = 2)

$w_i = (1 - \text{tox}_i)^{\gamma}$ — a more toxic comment contributes less to the sentiment aggregation.

**Why 2**
- Strategy performance is insensitive to γ (H-1, H-2): Sharpe 0.61–0.62, MDD fixed at −22.9% for γ 1/2/3 — it cannot have been chosen for performance.
- 2 is the principled midpoint of decay strength: γ=1 only halves a borderline comment (tox 0.5 → 0.5), γ=3 over-kills it (0.125). γ=2 → 0.25.
- Information loss is contained: effective-sample fraction 0.85 at γ=2 vs 0.84 at γ=3 (A-3).

### TOX

(left) The weighting function w = (1 - tox)^γ for γ ∈ {0, 1, 2, 3, 5}; at the chosen γ = 2, comments with toxicity 0.3, 0.5, 0.8 receive weights 0.49, 0.25, 0.04 respectively (black dots; red dashed line: hard-drop threshold τ = 0.95). (right) Share of the total daily-sentiment weight contributed by comments with toxicity ≥ x (2014-2025 sample). Because the γ = 2 weighting already suppresses toxic comments, those above 0.95 hold only 0.003% of the aggregate weight, so the daily sentiment index is effectively invariant to the hard-drop threshold.

In [ ]:
L.make_fig("TOX", R, LANG)

### A1

A more toxic comment gets a lower weight in the sentiment aggregation. Larger γ bends the curve down, attenuating even mid-toxicity comments quickly. At the current γ=2: toxicity 0.3 → weight 0.49, 0.5 → 0.25, 0.8 → 0.04. γ=1 (straight line) attenuates too weakly; γ≥3 discards too much.

In [ ]:
L.make_fig("A1", R, LANG)

### A2

The y-axis is how much the daily weighted-sentiment series differs, per day on average, from the no-weighting case. It is 0.013 at γ=2 and only 0.018 at γ=5. Given that the sentiment score ranges over [-1, 1], this is tiny: the daily index is nearly identical for any γ.

In [ ]:
L.make_fig("A2", R, LANG)

### A3

How many comments the weighting effectively uses, as a fraction of the total (1 = no loss). 0.85 at γ=2, 0.84 at γ=3. Most comments have low toxicity, so weighting costs under 15% of the sample and raising γ further adds little.

In [ ]:
L.make_fig("A3", R, LANG)

### A4

Top: the solid line is the monthly weighted sentiment at γ=2 (adopted); the shaded band is the range spanned by γ=1 and γ=3 (too narrow to see at this scale). Bottom: that same gap (max-min), plotted on its own axis — averaging 0.0037 and peaking at 0.0143, about 1% of the index's own swing (~0.6). The index barely moves for any γ between 1 and 3.

In [ ]:
L.make_fig("A4", R, LANG)

### A5

Most comments have low toxicity, so their weight sits near 1 regardless of γ. Larger γ moves more comments into the low-weight left tail, but they remain a small share of the whole.

In [ ]:
L.make_fig("A5", R, LANG)

## B. Toxicity hard-drop threshold τ  (current = 0.95)

`keep = tox < τ` — comments at or above the threshold are dropped entirely.

**Why 0.95**
- The index is structurally invariant to τ. The γ=2 weight already crushes toxic comments: those with tox ≥ 0.95 hold only **0.004%** of total sentiment weight (B-3). Lowering τ to 0.90 changes the daily index by MAD < 0.0002, to 0.80 by < 0.0004 (B-5).
- As a removal criterion, 0.95 is a conservative ceiling — it discards only what the model is ≥ 95% sure of (already 8.6% removed, B-1).
- Raising it lets clear toxicity through: the tox 0.95–0.99 band is mostly political invective, not sentiment signal (B-2).

### B1

Comments with score ≥ τ are dropped entirely. At the current 0.95, 8.6% (30,313 comments) are removed; 10.6% at 0.90, 5.1% at 0.99. Moving the threshold changes the removed volume only gradually.

In [ ]:
L.make_fig("B1", R, LANG)

### B2

Removed comments by band above the current threshold. The 0.99–1.00 band is largest (17,947). Raising the threshold to 0.97 or 0.99 would let the clearly toxic comments of these bands into the sentiment aggregation.

In [ ]:
L.make_fig("B2", R, LANG)

### B3

The key figure. Because of the γ=2 weighting, highly toxic comments carry almost no weight. All comments with toxicity ≥ 0.95 together hold just 0.003% of the total sentiment weight, and ≥ 0.90 only 0.015%. The daily index and strategy performance therefore barely change wherever τ is set.

In [ ]:
L.make_fig("B3", R, LANG)

### B4

The score is bimodal — near 0 (clean) and near 1 (clearly toxic) — with the ambiguous 0.4–0.9 range relatively sparse. As a result, the set of removed comments hardly changes wherever the threshold sits in that range.

In [ ]:
L.make_fig("B4", R, LANG)

### B5

Lowering the threshold from 0.95 to 0.90 moves the daily sentiment index by 0.00014 on average, and to 0.80 by 0.00032 — negligible on the [-1,1] scale even as many more comments are removed. (Sentiment probabilities exist only for kept comments, so only the 'stricter' direction is simulable; for the 'looser' direction see B-3.)

In [ ]:
L.make_fig("B5", R, LANG)

### B6

The hard-drop share ran 8.7-9.6% in 2014-2017, spiked to 13.2-13.7% in 2018-2019 (highlighted), then fell back to roughly 9% and, by 2023-2025, to 7-8%. Even the all-time peak (13.7%, 2018) stays under 15%.

In [ ]:
L.make_fig("B6", R, LANG)

## C. Keyword extraction length  (current = 2–6 characters)

This is a **rule justification**, not a sweep.

**Why 2–6**
- Lower bound 2: the only finance-relevant single-character token is '장'. Raising it to 3 loses 2-char core terms (기관/외인/환율/수급/시총/국장) (C-2).
- Upper bound 6: tokens of 7+ chars are inflected phrases already captured as substrings of shorter core keywords. The core-keyword set is identical for 6 vs 8 (C-2).

### C1

Bars: token occurrences by length. Red line (right axis): the share of those tokens matching the stock/economy pattern. Longer tokens are more often finance-related, but their count collapses after 4 characters. The yellow band marks the current extraction range (2–6 characters).

In [ ]:
L.make_fig("C1", R, LANG)

### C2

Number of words selected as core keywords under each length window. 2–6 equals 2–8 (raising the upper bound adds nothing), while 3–6 drops sharply (losing 2-character financial terms). Lower bound 1 is essentially the same as 2 (almost no single-character financial words).

In [ ]:
L.make_fig("C2", R, LANG)

## D. Core-keyword DF threshold  (real run = DF 0.5%)

**Why around 0.5%**
- Lowering the DF threshold adds keywords but mixes in generic words ('가장', '당장'); raising it is cleaner but loses valid compounds (base rate).
- 1% is the balance point, and its pass rate (29%) matches the actual output file `comments_stock_clean`.

The real run used `min_doc_ratio=0.005` (DF 0.5%); F and G use the stock_score / is_stock columns from that run.

### D1

A word is a core keyword if its document frequency is at least the threshold, and an auxiliary keyword in the 30–100% band below it. Raising the threshold shrinks the set quickly (annual mean core keywords: ~6 at 0.5%, ~2 at 1%). At 1%, every year keeps only 'short-selling / stock / interest rate', dropping 'share price / KOSPI / exchange rate'. Real run value: 0.5% (40k-per-year stratified sample).

In [ ]:
L.make_fig("D1", R, LANG)

### D2

Effect of the DF threshold on the pass rate (re-extraction on the stratified sample): ~13% at 0.5%, ~7% at 1%, ~2% at 2%. Above 1%, the number of core keywords collapses to 0–2 per year. The real run used 0.5%; the F/G pass rate (15.1%) uses that run's actual stock_score / is_stock columns (full 2014-2025).

In [ ]:
L.make_fig("D2", R, LANG)

### D3

An auxiliary keyword sits in the band 'DF threshold × multiplier ... DF threshold'. The current 0.3 means 0.3%–1%. Lowering the multiplier to 0.1 sharply increases the auxiliary-keyword count (adding rare words) but barely moves the pass rate, because the core keywords already decide passage. 0.3 keeps auxiliary keywords to 'frequent but below core' words.

In [ ]:
L.make_fig("D3", R, LANG)

## F · G · E. Score cutoff · core ≥ 1 condition · 10 : 3 weighting

score = 10·(core keyword count) + 3·(auxiliary keyword count),  pass = score ≥ 10 **and** core ≥ 1.

**Why this design**
- **score ≥ 10** is the end of the pass-rate plateau. Cutoffs 3·5·7·10 all give 15.1%; at 13 it drops to 10.6% (F-1). The drop is the comments with exactly one core keyword (score 10) being cut.
- **10 : 3** — changing the aux weight to 2·5·10 leaves the passing set 100% identical. One core keyword = 10 already clears the gate; the aux weight only ranks.
- **core ≥ 1** — without it, ~1,858 comments (1.0% of passers) get through on auxiliary keywords alone.

### F1

The key figure. score = 10·(core keyword count) + 3·(auxiliary keyword count). For cutoffs 3, 5, 7, 10 the pass rate is identical at 15.1% (shaded band); at 13 it drops sharply to 10.6%. That drop is the comments with exactly one core keyword (score 10) being cut. 10 is the end of the plateau.

In [ ]:
L.make_fig("F1", R, LANG)

### F2

A large peak at score 0 (not relevant) and a distinct second peak at score 10 (one core keyword). Cutoff 10 includes this second peak; cutoff 13 excludes it.

In [ ]:
L.make_fig("F2", R, LANG)

### E1

Passage is decided by whether one core keyword (= the core score) clears the gate (cutoff 10). With a core score of 10, changing the auxiliary score to 2·3·5·10 leaves the pass rate unchanged; lowering the core score to 5 or 3 means one core keyword no longer clears the gate and the pass rate collapses (to 1.7% and 0.7%). So 'core = 10' is essential; 'auxiliary = 3' only ranks the comments that pass.

In [ ]:
L.make_fig("E1", R, LANG)

### G1

Requiring only score ≥ 10 gives a 15.25% pass rate; adding 'at least one core keyword' gives 15.10%. The difference is the comments that cleared score 10 on '4+ auxiliary keywords, 0 core' — 1,858 (1.0% of passers). These are social/political posts where 'market/investment' appear in passing, without a financial core keyword, and this condition removes them.

In [ ]:
L.make_fig("G1", R, LANG)

## Strategy performance (10-year K-FGI pipeline, re-derived)

K-FGI → position → strategy return re-derived over the full period for varying γ and sentiment MA window. Code: `downstream_strategy_sensitivity.py`; numbers: `downstream_sensitivity.csv`.

**Key** — γ has no effect on strategy performance (H-1, H-2); the 10-day sentiment MA is a clear local optimum (I-1). So τ and γ are 'anything in the neighbourhood is fine', while the sentiment MA 'has to be 10 days'.

### H1

Result of re-deriving comment → daily sentiment features → K-FGI → position → strategy return over the full 10 years for γ = 1, 2, 3. Sharpe 0.610 / 0.613 / 0.622 and Calmar near 0.36 — essentially flat. γ=2 is not a performance optimum.

In [ ]:
L.make_fig("H1", R, LANG)

### H2

MDD −22.9%, CVaR (5%) −2.13%, downside volatility 0.114 — unchanged to three decimals from γ 1 to 3. The toxicity weighting exponent does not affect the strategy's downside risk.

In [ ]:
L.make_fig("H2", R, LANG)

### I1

Strategy performance when sent_composite is smoothed over 5, 10 or 20 trading days. 10 days is a clear local optimum on Sharpe (0.61), Calmar (0.36) and MDD (−22.9%). 5 days is noisy, 20 days lags. (Unlike τ and γ, this value does move the result, so it needs an explicit rationale.)

In [ ]:
L.make_fig("I1", R, LANG)

## Conclusion — convention vs. our choice

| Value | Type | Verdict |
|---|---|---|
| τ = 0.95 | our choice | ✓ index & strategy structurally invariant to τ (tox≥0.95 weight mass 0.003%) |
| γ = 2 | our choice | ✓ no performance effect; principled midpoint of decay strength |
| score ≥ 10 | our choice | ✓ end of the pass-rate plateau (3–10) |
| DF ≥ 0.5% | our choice | ◐ excludes generic words; 1%+ collapses to 0–2 core keywords/year |
| core 10 : aux 3 | our choice | ◐ aux weight irrelevant to the gate, ranking only |
| core ≥ 1 | our choice | ✓ blocks the few comments passing on aux keywords alone |
| 2–6 characters | method rule | ✓ keeps 2-char financial terms / long phrases caught as substrings |
| 10-day sentiment MA | our choice | ✓ local optimum on Sharpe and MDD (a value that does need a rationale) |
| 252-day · 60-day · 70th pct. | finance convention | not tuned |